In [ ]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
candidate_paths = [
    Path.cwd() / "results" / "initial_plan.json",
    Path.cwd().parent / "ds_job_interview_prep" / "results" / "initial_plan.json",
    Path("ds_job_interview_prep/results/initial_plan.json"),
]
initial_plan_path = next(
    (p for p in candidate_paths if p.is_file()),
    Path("ds_job_interview_prep/results/initial_plan.json"),
)
if not initial_plan_path.exists():
    raise FileNotFoundError(
        f"Could not find initial_plan.json. Checked: {[str(p) for p in candidate_paths]}"
    )
initial_plan_path

In [ ]:

initial_plan_path = Path.cwd() / "results" / "initial_plan.json"
with initial_plan_path.open("r", encoding="utf-8") as f:
    initial_plan = json.load(f)

revised_plan_path = Path.cwd() / "results" / "revised_plan.json"
with revised_plan_path.open("r", encoding="utf-8") as f:
    revised_plan = json.load(f)

final_plan_path = Path.cwd() / "results" / "final_plan.json"
with final_plan_path.open("r", encoding="utf-8") as f:
    final_plan = json.load(f)

In [ ]:

def get_plan_text(value, day="Day 1"):
    """Extract the requested day's section from common API response structures."""
    if not re.fullmatch(r"Day\s+\d+", day.strip(), re.IGNORECASE):
        raise ValueError("day must look like 'Day 1' or 'Day 2'")

    def to_text(item):
        if isinstance(item, str):
            return item
        if isinstance(item, dict):
            for key in ("content", "text", "output_text"):
                if key in item:
                    text = to_text(item[key])
                    if text:
                        return text
            return "\n".join(filter(None, (to_text(v) for v in item.values())))
        if isinstance(item, (list, tuple)):
            return "\n".join(filter(None, (to_text(v) for v in item)))
        return ""

    text = to_text(value)
    heading_pattern = re.compile(
        r"(?im)^\s{0,3}(?:#{1,6}\s*)?(?:\d+[.)]\s*)?(?:\*\*)?(Day\s+\d+)\b.*$"
    )
    headings = list(heading_pattern.finditer(text))
    requested_day = re.sub(r"\s+", " ", day.strip()).casefold()

    for index, heading in enumerate(headings):
        found_day = re.sub(r"\s+", " ", heading.group(1)).casefold()
        if found_day == requested_day:
            end = headings[index + 1].start() if index + 1 < len(headings) else len(text)
            return text[heading.start():end].strip()

    raise ValueError(f"{day!r} was not found in the response")

In [ ]:
initial_plan_text = get_plan_text(initial_plan['choices'][0]['message']['content'], day="Day 1")
display(Markdown(initial_plan_text))

In [ ]:
revised_plan_text = get_plan_text(revised_plan['choices'][0]['message']['content'], day="Day 1")
display(Markdown(revised_plan_text))

In [ ]:
final_plan_text = get_plan_text(final_plan['choices'][0]['message']['content'], day="Day 1")
display(Markdown(final_plan_text))